### Laboratorio 11: coordinación, fallos y HotpotQA real

Duración: cuatro horas.

Trabajaremos primero sobre las 7.405 preguntas originales. Después ejecutaremos pruebas controladas de contratos y analizaremos un piloto LLM solo si los recursos de Docker lo permiten.

Entrega un informe breve con IDs de consultas reales, métricas reproducibles, trazas y amenazas a la validez. No fabriques respuestas ni casos fallidos que no aparecieron.

#### 1. Entorno y datos verificables

Ejecuta este laboratorio desde `/workspace/CC-0F4` con HotpotQA preparado previamente. Si falta el archivo o el hash no coincide, corrige la descarga. No uses datos sustitutivos para obtener un resultado positivo.

In [ ]:
from pathlib import Path
import os, json, sys, csv, collections, copy
root = Path.cwd().resolve()
for candidate in (root, *root.parents):
    if (candidate / 'Semana11/multiagente_real.py').is_file():
        root = candidate
        break
else:
    raise FileNotFoundError('Abre JupyterLab dentro de CC-0F4')
if str(root) not in sys.path:
    sys.path.insert(0,str(root))
from Semana11.multiagente_real import (
    check_dataset, ranked_evidence, gold_facts, support_recall, retrieval_rows,
    summarize_retrieval, evaluate_retrieval, validate_answer, select_pilot, pilot, MODEL_ID)
from Semana11.contratos import Blackboard, ContractError, make_message, validate_selection
examples = check_dataset()
assert len(examples) == 7405
print('HotpotQA verificado:', len(examples))

#### 2. Lectura de un problema real

Selecciona una pregunta directamente del dataset. Anota su identificador, el tipo de pregunta, sus párrafos disponibles y la estructura de las anotaciones humanas. El código no necesita crear preguntas nuevas.

In [ ]:
ex = examples[0]
print('ID:',ex['_id'])
print('Pregunta:',ex['question'])
print('Tipo:',ex['type'],'| nivel:',ex['level'])
print('Párrafos disponibles:',len(ex['context']))
print('Oraciones de apoyo:',len(ex['supporting_facts']))

#### 3. Ranking y trazabilidad

Calcula top-14 con BM25, registra título, índice de oración y puntuación. Justifica por qué el recuperador no debe recibir `supporting_facts`. Repite la recuperación y verifica que el orden sea idéntico.

In [ ]:
observed = ranked_evidence(ex,14)
assert observed == ranked_evidence(ex,14)
for item in observed[:7]:
    print(item['id'], item['title'], item['sent_id'],
          f"BM25={item['bm25']:.4f}",item['text'][:115])

#### 4. Métricas de evidencia, no de respuesta

Calcula cuántos hechos gold aparecen entre las oraciones seleccionadas. Compara AnyHit y AllHit. Responde por qué una pregunta con AnyHit uno todavía puede no tener toda la evidencia de varios saltos.

In [ ]:
gold = gold_facts(ex)
found = {(i['title'],i['sent_id']) for i in observed}
print('Gold:',len(gold),'intersección:',len(gold & found))
print('Recall:',support_recall(ex,observed))
print('AnyHit:',bool(gold & found),'AllHit:',bool(gold and gold <= found))

#### 5. Evaluación obligatoria de las 7.405 preguntas

Este paso produce filas reales por pregunta. Guarda el resumen y verifica el total antes de interpretar los promedios. El resultado del laboratorio no depende de una muestra de ocho o quince preguntas.

In [ ]:
summary = evaluate_retrieval()
assert summary['questions']==7405
print(json.dumps(summary['retrieval'],ensure_ascii=False,indent=2))
rows_path = root/'Semana11/resultados/hotpot_recuperacion_7405_por_consulta.jsonl'
rows = [json.loads(line) for line in rows_path.read_text(encoding='utf-8').splitlines()]
assert len(rows)==7405
print('Trazabilidad:',len(rows),'filas')

#### 6. Fallos empíricos de recuperación

Busca preguntas con AllHit igual a cero, registrando sus IDs, tipo y número de hechos gold. No uses preguntas inventadas. Si no existe un grupo solicitado, indica que no aparece en esta ejecución.

In [ ]:
failures = [row for row in rows if row['all_support'] == 0]
print('Preguntas sin recuperación completa:',len(failures))
for row in failures[:5]:
    print(row['question_id'],row['type'],row['level'],
          'gold',row['n_gold'],'hallados',row['n_support_found'])

#### 7. Comparación por tipo de pregunta

Contrasta `bridge` y `comparison` si ambos aparecen. Explica por qué el tipo de pregunta puede modificar la necesidad de dos fragmentos de evidencia. No atribuyas automáticamente estas diferencias a la arquitectura multiagente.

In [ ]:
for typ,values in summary['by_type'].items():
    print(typ,'N=',values['questions'],
          'AnyHit=',round(values['any_support_rate'],4),
          'AllHit=',round(values['all_support_rate'],4))

#### 8. Contrato de mensajes y pizarra

Construye un mensaje a partir de los identificadores realmente recuperados y verifica que el supervisor lo acepta. Comprueba que cambiar después el objeto original no modifica el contenido ya registrado.

In [ ]:
board=Blackboard(ex['_id'])
ids=[observed[0]['id'], observed[1]['id']]
message=make_message(ex['_id'],'evidence_worker',ids,observed)
board.append(message,observed)
message['selected_ids'].clear()
print('Pizarra validada:',board.messages[0]['selected_ids'])
assert board.messages[0]['selected_ids']==ids

#### 9. Fallos de ingeniería inyectados

Los siguientes errores son pruebas de contratos y deben mantenerse fuera de las métricas de HotpotQA. El objetivo es explicar el mecanismo de rechazo, no simular un agente real que falla.

In [ ]:
# Se registran motivos de rechazo en los contratos, no métricas sintéticas.
cases = [
    ('cita_no_observada', lambda: validate_selection(['E99'],observed)),
    ('citas_duplicadas', lambda: validate_selection(['E00','E00'],observed)),
    ('seleccion_vacia', lambda: validate_selection([],observed)),
    ('rol_no_autorizado', lambda: make_message(ex['_id'],'admin',['E00'],observed)),
]
for label, func in cases:
    try:
        func()
    except ContractError as err:
        print(label, 'rechazado:',err)
    else:
        raise AssertionError('Se aceptó el fallo '+label)

#### 10. Procedencia y huellas de texto

La comprobación de identificadores puede ser insuficiente si alguien modifica un fragmento después de la recuperación. La huella SHA-256 debe cubrir contenido y localizador. Comprueba el rechazo de una huella alterada.

In [ ]:
invalid=copy.deepcopy(board.messages[0])
invalid['evidence_sha256'][ids[0]]='hash_incorrecto'
try:
    board.append(invalid,observed)
except ContractError as err:
    print('Procedencia alterada detectada:',err)
else:
    raise AssertionError('El contrato aceptó un hash falso')

#### 11. Contrato de respuesta final

Distingue una finalización estructural válida de una respuesta semánticamente correcta. La prueba utiliza el formato del contrato, no una afirmación sobre la respuesta humana a la pregunta.

In [ ]:
ok,reason,_,cites=validate_answer({'answer':'Revisar evidencia original',
                                           'cited_ids':[ids[0]]},observed)
print(ok,reason,cites)
assert ok
invalid,invalid_reason,_,_=validate_answer({'answer':'texto','cited_ids':['E99']},observed)
print(invalid,invalid_reason)
assert not invalid

#### 12. Pruebas unitarias del controlador

Ejecuta los tests sin red ni Qwen. Las entradas pequeñas de los tests sirven para verificar contratos; nunca se mezclan con las 7.405 preguntas auténticas que componen el benchmark.

In [ ]:
import subprocess
result=subprocess.run([sys.executable,'-m','unittest','discover','-s',
                       'Semana11/tests','-t','.','-p','test_*.py','-v'],
                      cwd=root,capture_output=True,text=True)
print(result.stdout[-3000:]); print(result.stderr[-3000:])
assert result.returncode==0

#### 13. Piloto opcional con inferencia LLM real

En el modo opcional se usa el mismo Qwen para ambas condiciones. El agente único necesita una llamada y el supervisor con trabajadores utiliza una o dos según los fallos. No hay resultados garantizados, y las respuestas JSON inválidas se registran sin repararlas.

In [ ]:
if os.getenv('CC0F4_RUN_REAL_LLM','0')=='1':
    count=int(os.getenv('CC0F4_S11_LLM_N','16'))
    report=pilot(count,os.getenv('CC0F4_S11_MODEL',MODEL_ID))
    print(json.dumps(report,ensure_ascii=False,indent=2))
else:
    print('LLM no ejecutado. La práctica obligatoria de 7405 preguntas sí está evaluada.')

#### 14. Análisis de resultados y defensa

En tu informe indica: el número real de preguntas, AllHit y Recall medios, al menos dos IDs de error del benchmark, la razón de rechazo de tres mensajes malformados y una limitación de la comparación multiagente. Si ejecutaste el piloto, aporta EM, F1, tasas de finalización válida, tokens y fallos por condición.

No uses una finalización estructural válida como prueba de factualidad. No prometas que un supervisor con dos trabajadores supera a un agente único cuando la evidencia no lo demuestra.

#### 15. Entrega

Adjunta el notebook ejecutado, el resumen de recuperación, una selección de trazas reales de error, el resultado de las pruebas unitarias y un comentario crítico de hasta dos páginas. Si no existe inferencia LLM real, indica expresamente que el piloto queda pendiente.

Identifica las rutas y los hashes de datos. No cambies preguntas ni respuestas humanas para producir métricas favorables.